# 03a — REHAB24-6: streaming features (cells for `benchmark_3d_world.ipynb` on Colab)

**What this does.** STEP 5 of the benchmark built the I_ML features from the *whole*
recording at once. Here the same recordings are fed to `iml_live.stream_features()`
**one frame at a time, as a webcam would deliver them**, so each row only uses the
frames before it. Both versions are saved side by side.

**Order**
1. Push the repo with `iml_live.py` (FeatureStream / stream_features) to GitHub.
2. In Colab: Runtime → Restart, then run STEP 1 → STEP 5 of `benchmark_3d_world.ipynb` (the landmark cache is reused, so this is quick).
3. Paste **cell S1** after STEP 5 and run it (≈ 5–15 min: one feature row per frame).
4. Paste **cell S2** after it and run it. It writes `feat_rehab24_stream.npz` to `RESULTS`.
5. Download that file into the repo's `data/fit3d/` folder on your PC.

In [ ]:
# ===== CELL S1 - replay every kept recording frame by frame (paste after STEP 5) =====
import time, importlib
import numpy as np
import iml_live; importlib.reload(iml_live)
from iml_live import stream_features
from features import overlap_rows

for _need in ('aligned_cache', 'recordings', 'get_3d_landmarks', 'THRESHOLD'):
    assert _need in dir(), f'{_need} missing - run STEP 1 to STEP 5 first'

rec_by_id = {r['rec_id']: r for r in recordings}
by_rec = {}
for it in aligned_cache:
    rid, side = it['meta']['recording'].rsplit('_', 1)     # recording = "<rec_id>_<side>"
    by_rec.setdefault(rid, []).append((side, it))

t0, done, bad = time.time(), 0, []
for k, (rid, items) in enumerate(by_rec.items(), 1):
    lm, world, size, fps = get_3d_landmarks(rec_by_id[rid])          # cached landmarks, raw
    for side, it in items:
        S = stream_features(lm, world, side, size, fps, threshold=THRESHOLD)   # (frames, 20)
        # same lag as STEP 5, so row i still pairs with the same truth sample
        S_al, _ = overlap_rows(S, np.zeros(len(world)), it['meta']['lag'])
        if S_al.shape != np.asarray(it['feat']).shape:
            bad.append((it['meta']['recording'], S_al.shape, np.asarray(it['feat']).shape)); continue
        it['feat_stream'] = S_al
        done += 1
    print(f'[{k}/{len(by_rec)}] {rid}  {time.time() - t0:.0f}s', flush=True)

print(f'\nstreamed {done} arm-recordings, {len(bad)} shape mismatches')
for b in bad[:10]:
    print('  mismatch', *b)

In [ ]:
# ===== CELL S2 - save offline AND streaming features side by side =====
import os, inspect
import numpy as np
import benchmark_common as bc
from features import FEATURE_NAMES

I2D = FEATURE_NAMES.index('angle_2d')
parts = []
for it in aligned_cache:
    if 'feat_stream' not in it:
        continue
    me = it['meta']
    F, S, t = (np.asarray(it[k], float) for k in ('feat', 'feat_stream', 'truth_3d'))
    n = min(len(F), len(S), len(t)); F, S, t = F[:n], S[:n], t[:n]
    ok = np.isfinite(t) & (np.isfinite(F[:, I2D]) | np.isfinite(S[:, I2D]))
    if ok.any():
        parts.append((F[ok], S[ok], t[ok], me))

n = [len(p[2]) for p in parts]
out = os.path.join(RESULTS, 'feat_rehab24_stream.npz')
np.savez_compressed(
    out,
    X_offline=np.concatenate([p[0] for p in parts]).astype(np.float32),
    X_stream=np.concatenate([p[1] for p in parts]).astype(np.float32),
    y=np.concatenate([p[2] for p in parts]).astype(np.float32),
    feature_names=np.array(FEATURE_NAMES),
    person=np.repeat(['rehab_' + str(p[3]['person']) for p in parts], n),
    recording=np.repeat(['rehab_' + str(p[3]['recording']) for p in parts], n),
    view=np.repeat([str(p[3]['view']) for p in parts], n),
    exercise=np.repeat([str(p[3]['ex']) for p in parts], n),
    dataset=np.array(['rehab24'] * sum(n)),
    causal=np.array('lfilter_zi' in inspect.getsource(bc.butterworth_filter)),
)
print(f'saved {out}: {sum(n):,} frames, {len(parts)} arm-recordings, '
      f'{len(set(p[3]["person"] for p in parts))} people')